In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import pandas as pd
import numpy as np

load_dotenv(override=True)

url = URL.create(
    "postgresql+psycopg2",
    username=os.environ.get("DB_USER", "postgres"),
    password=os.environ["DB_PASSWORD"],
    host=os.environ.get("DB_HOST", "localhost"),
    port=os.environ.get("DB_PORT", "5432"),
    database=os.environ.get("DB_NAME", "finguard_db"),
)
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM transactions", engine)
print(df.shape)
df.head()

In [ ]:
df['Hour'] = (df['Time'] // 3600) % 24

In [ ]:
# Flag transactions happening in the high-fraud-rate window (hours 1-5)
df['Is_High_Risk_Hour'] = df['Hour'].isin([1, 2, 3, 4, 5]).astype(int)

print(df['Is_High_Risk_Hour'].value_counts())
print(df.groupby('Is_High_Risk_Hour')['Class'].mean() * 100)

Is_High_Risk_Hour
0    268568
1     16239
Name: count, dtype: int64
Is_High_Risk_Hour
0    0.139257
1    0.726646
Name: Class, dtype: float64


In [ ]:
# Log-transform Amount (raw amount is heavily right-skewed — log helps models learn better)
df['Amount_Log'] = np.log1p(df['Amount'])  # log1p handles Amount=0 safely

# Z-score of amount (how many std deviations from the mean)
df['Amount_Zscore'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()

# Is this a "round number" amount? (fraud sometimes uses suspiciously round test amounts)
df['Is_Round_Amount'] = (df['Amount'] % 1 == 0).astype(int)

print(df[['Amount', 'Amount_Log', 'Amount_Zscore', 'Is_Round_Amount']].head(10))

   Amount  Amount_Log  Amount_Zscore  Is_Round_Amount
0  149.62    5.014760       0.244964                0
1    2.69    1.305626      -0.342474                0
2  378.66    5.939276       1.160684                0
3  123.50    4.824306       0.140534                0
4   69.99    4.262539      -0.073403                0
5    3.67    1.541159      -0.338556                0
6    4.99    1.790091      -0.333278                0
7   40.80    3.732896      -0.190107                0
8   93.20    4.545420       0.019392                0
9    3.68    1.543298      -0.338516                0


In [ ]:
# Faster approach using searchsorted (vectorized, no full-table apply)
time_vals = df['Time'].values
window_start = time_vals - 3600  # 1 hour = 3600 seconds

# For each transaction, count how many transactions fall in [time - 3600, time)
lower_idx = np.searchsorted(time_vals, window_start, side='left')
upper_idx = np.searchsorted(time_vals, time_vals, side='left')

df['Txn_Count_Last_Hour'] = upper_idx - lower_idx

print(df[['Time', 'Txn_Count_Last_Hour', 'Class']].head(10))
print("\nAvg txn count in last hour - Legit vs Fraud:")
print(df.groupby('Class')['Txn_Count_Last_Hour'].mean())

   Time  Txn_Count_Last_Hour  Class
0   0.0                    0      0
1   0.0                    0      0
2   1.0                    2      0
3   1.0                    2      0
4   2.0                    4      0
5   2.0                    4      0
6   4.0                    6      0
7   7.0                    7      0
8   7.0                    7      0
9   9.0                    9      0

Avg txn count in last hour - Legit vs Fraud:
Class
0    7246.302478
1    6082.847561
Name: Txn_Count_Last_Hour, dtype: float64


In [ ]:
# Drop columns we no longer need in raw form
# (keep Amount for reference in EDA, but Amount_Log is what we'll feed models)
df_final = df.drop(columns=['Time'])  # raw Time no longer needed, we extracted Hour from it

print(df_final.columns.tolist())
print(df_final.shape)

['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class', 'Hour', 'Is_High_Risk_Hour', 'Amount_Log', 'Amount_Zscore', 'Is_Round_Amount', 'Txn_Count_Last_Hour']
(284807, 36)


In [ ]:
# Save locally as CSV (for fast reloading in later notebooks)
df_final.to_csv("../data/processed/transactions_features.csv", index=False)
print("Saved to data/processed/transactions_features.csv")

# Also push to PostgreSQL as a new table
df_final.to_sql("transactions_features", engine, if_exists="replace", index=False, chunksize=10000)
print("Saved to PostgreSQL as 'transactions_features'")

Saved to data/processed/transactions_features.csv
Saved to PostgreSQL as 'transactions_features'
